In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import gzip
import warnings
warnings.filterwarnings('ignore')

print("Loading data...")

columns = [
    'karl_id', 'host_name', 'model_name', 'hardware_make', 'karl_last_seen',
    'auth_username', 'serial_number', 'group_id', 'tenant_id', 'platform',
    'metric_category', 'measure_name', 'time', 'p90_processor_time',
    'avg_processor_time', 'max_cpu_usage', 'p90_memory_utilization',
    'avg_memory_utilization', 'max_memory_usage', 'p10_battery_health',
    'avg_battery_health', 'cpu_count', 'memory_count', 'memory_size_gb',
    'driver_vendor', 'os', 'wifi_mac_add', 'driver_version', 'driver_date',
    'os_version', 'driver', 'agent_id', 'performance_status', 'device_status',
    'max_battery_temperature', 'avg_battery_temperature', 'p90_battery_temperature',
    'avg_cpu_temp', 'p90_cpu_temp', 'avg_battery_discharge', 'p90_battery_discharge',
    'avg_boot_time', 'p90_boot_time', 'uptime_days', 'total_app_crash'
]

# Load sample
chunk_size = 100000
sample_data = []
with gzip.open('000.gz', 'rt') as f:
    for i, chunk in enumerate(pd.read_csv(f, sep='|', names=columns, chunksize=chunk_size)):
        sample_data.append(chunk)
        if i >= 4:
            break

df = pd.concat(sample_data, ignore_index=True)

numeric_cols = [
    'avg_processor_time', 'max_cpu_usage', 'avg_memory_utilization',
    'max_memory_usage', 'avg_battery_health', 'cpu_count', 'memory_size_gb',
    'avg_cpu_temp', 'avg_boot_time', 'p90_boot_time',
    'uptime_days', 'total_app_crash'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Loaded {len(df)} rows")

    

Loading data...
Loaded 500000 rows


In [6]:
feature_cols = ['avg_processor_time','max_cpu_usage','avg_memory_utilization', 'max_memory_usage',
'avg_battery_health','cpu_count','memory_size_gb','avg_cpu_temp','avg_boot_time','p90_boot_time',
'uptime_days']

# LOOP THROUGH FEATURE_COLS 

results = {col: 0 for col in feature_cols}

for col in feature_cols:
    model_df = df[feature_cols].dropna()
    X = model_df.drop(col, axis=1)
    y = model_df[col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    print("\nTraining Random Forest Regressor...")
    print(col)

    rf_model = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)

    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("\n=== PREDICTION PERFORMANCE ===")
    print(f"MSE: {mse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"R²: {r2:.3f}")

    remaining_features = [f for f in feature_cols if f != col]

    importance = pd.DataFrame({
        'feature': remaining_features,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)

    # accumulate importance
    for feat, imp in zip(remaining_features, rf_model.feature_importances_):
        results[feat] += imp

    print("\nTop Predictors of {col}")
    print(importance)

    high_mem_threshold = model_df[col].quantile(0.90)
    actual_high_risk = y_test >= high_mem_threshold
    pred_high_risk = y_pred >= high_mem_threshold
    risk_accuracy = (actual_high_risk == pred_high_risk).mean()

# AFTER the loop
results_avg = {k: v / (len(feature_cols) - 1) for k, v in results.items()}
# Use the average
results_df = pd.DataFrame(
    results_avg.items(),
    columns=["feature", "avg_importance"]
).sort_values("avg_importance", ascending=False)

top3 = results_df[:3]

print("Top 3 Root Cause Predictions:")
print(top3)



Training Random Forest Regressor...
avg_processor_time

=== PREDICTION PERFORMANCE ===
MSE: 4.23
MAE: 1.31
R²: 0.946

Top Predictors of {col}
                  feature  importance
0           max_cpu_usage    0.959925
1  avg_memory_utilization    0.010054
4               cpu_count    0.008649
2        max_memory_usage    0.004327
6            avg_cpu_temp    0.003789
9             uptime_days    0.003714
7           avg_boot_time    0.003025
5          memory_size_gb    0.002664
3      avg_battery_health    0.002289
8           p90_boot_time    0.001562

Training Random Forest Regressor...
max_cpu_usage

=== PREDICTION PERFORMANCE ===
MSE: 16.79
MAE: 2.54
R²: 0.935

Top Predictors of {col}
                  feature  importance
0      avg_processor_time    0.958031
4               cpu_count    0.013480
1  avg_memory_utilization    0.005999
2        max_memory_usage    0.004801
3      avg_battery_health    0.003917
6            avg_cpu_temp    0.003864
9             uptime_days    0.003